Load the "MainProcess.xes" file

In [13]:
import os
from pm4py.objects.log.importer.xes import importer as xes_importer
import pandas as pd

# Directory with your .xes files
xes_directory = os.path.join(os.getcwd(), "20130794", "Cleaned Event Log")

# Output directory for models or visuals
output_directory = os.path.join(os.getcwd(), "output")
os.makedirs(output_directory, exist_ok=True)

# Process MainProcess.xes file
filename = "MainProcess.xes"
file_path = os.path.join(xes_directory, filename)
print(f"Processing {filename}")

try:
    log = xes_importer.apply(file_path)
    print("\n\n")
    print(f"Imported {filename} with {len(log)} traces.")
    
    # Print important information about the log
    print(f"Number of events: {sum(len(trace) for trace in log)}")
    activities = set(event["concept:name"] for trace in log for event in trace if "concept:name" in event)
    print(f"Number of unique activities: {len(activities)}")
    print(f"Unique activities: {activities}")
    case_ids = [trace.attributes["concept:name"] for trace in log if "concept:name" in trace.attributes]
    print(f"Number of cases: {len(case_ids)}")
    print(f"First 5 case IDs: {case_ids[:5]}")
    print("\n\n")

except Exception as e:
    print(f"Error processing {filename}: {e}")



Processing MainProcess.xes


parsing log, completed traces :: 100%|██████████| 301/301 [00:00<00:00, 670.24it/s]




Imported MainProcess.xes with 301 traces.
Number of events: 9471
Number of unique activities: 21
Unique activities: {'/dm/lower', '/mm/transport_from_to', '/hbw/store_empty_bucket', '/hbw/unload', '/pm/punch_ribbing', '/sm/sort', '/ov/temper', '/mm/drill', '/vgr/pick_up_and_transport', '/dm/cylindrical_drill', '/hbw/get_empty_bucket', '/hbw/store', '/hw/human_review', '/pm/punch_recesses', '/sm/transport', '/pm/punch_gill', '/dm/drill', '/mm/mill', '/mm/deburr', '/ov/burn', '/wt/pick_up_and_transport'}
Number of cases: 301
First 5 case IDs: ['WF_101_0', 'WF_102_0', 'WF_103_0', 'WF_104_0', 'WF_105_0']





Get all the attributes included in this file ("MainProcess.xes")

In [14]:
try:
    # List all attributes of the traces in a table
    print("\n\nAttributes of traces:\n")
    
    trace_attributes = set()
    for trace in log:
        trace_attributes.update(trace.attributes.keys())
        
    trace_attr_info = []
    for attr in trace_attributes:
        attr_type = "unknown"
        for trace in log:
            if attr in trace.attributes:
                attr_type = type(trace.attributes[attr]).__name__
                break
        trace_attr_info.append({"Attribute": attr, "Type": attr_type})

    trace_attr_df = pd.DataFrame(trace_attr_info)
    display(trace_attr_df)
    # Save the attribute DataFrame to an Excel file in a "tables" subfolder
    tables_dir = os.path.join(os.getcwd(), "tables")
    os.makedirs(tables_dir, exist_ok=True)
    trace_attr_df.to_excel(os.path.join(tables_dir, "main_trace_attribute_info.xlsx"), index=False)
    
    # List all attributes of the events in a table
    print("\n\nAttributes of events:\n")
    
    event_attributes = set()
    for trace in log:
        for event in trace:
            event_attributes.update(event.keys())

    event_attr_info = []
    for attr in event_attributes:
        attr_type = "unknown"
        for trace in log:
            for event in trace:
                if attr in event:
                    attr_type = type(event[attr]).__name__
                    break
            if attr_type != "unknown":
                break
        event_attr_info.append({"Attribute": attr, "Type": attr_type})

    event_attr_df = pd.DataFrame(event_attr_info)
    display(event_attr_df)
    # Save the attribute DataFrame to an Excel file in a "tables" subfolder
    tables_dir = os.path.join(os.getcwd(), "tables")
    os.makedirs(tables_dir, exist_ok=True)
    event_attr_df.to_excel(os.path.join(tables_dir, "main_event_attribute_info.xlsx"), index=False)
except Exception as e:
    print(f"Error processing the log: {e}")




Attributes of traces:



,Attribute,Type
0,concept:name,str




Attributes of events:



,Attribute,Type
0,human_workstation_green_button_pressed,float
1,concept:name,str
2,parameters,dict
3,event_id,str
4,lifecycle:transition,str
5,time:timestamp,datetime
6,lifecycle:state,str
7,unsatisfied_condition_description,str
8,case:concept:name,str
9,process_model_id,str


List all the resources of the log in a table

In [15]:
try:
    # Set pandas display options for full width
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.width', 0)
    pd.set_option('display.max_columns', None)

    resources = set()
    for trace in log:
        for event in trace:
            if "org:resource" in event:
                resources.add(event["org:resource"])

    resource_info = []
    for resource in resources:
        # Find the first SubProcessID for this resource by searching events until found
        first_subprocess_id = "N/A"
        parameters_dict = None
        found = False
        for trace in log:
                for event in trace:
                    if event.get("org:resource") == resource:
                        if "SubProcessID" in event:
                            first_subprocess_id = event["SubProcessID"]
                            found = True
                            break
                if found:
                    break
                
        found = False
        for trace in log:
                for event in trace:
                    if event.get("org:resource") == resource:
                        if "parameters" in event:
                            parameters_dict = event["parameters"]
                            found = True
                            break
                if found:
                    break
        event_count = sum(1 for trace in log for event in trace if event.get("org:resource") == resource)
        activities_performed = set(event["concept:name"] for trace in log for event in trace if event.get("org:resource") == resource)
        
        resource_info.append({
            "Resource": resource,
            "Event Count": event_count,
            "Unique Activities": len(activities_performed),
            "Activities": ", ".join(sorted(activities_performed)),
        })

    resource_df = pd.DataFrame(resource_info)
    display(resource_df)
    # Save the attribute DataFrame to an Excel file in a "tables" subfolder
    tables_dir = os.path.join(os.getcwd(), "tables")
    os.makedirs(tables_dir, exist_ok=True)
    resource_df.to_excel(os.path.join(tables_dir, "resources_info.xlsx"), index=False)

except Exception as e:
    print(f"Error processing the log: {e}")

,Resource,Event Count,Unique Activities,Activities
0,wt_1,447,1,/wt/pick_up_and_transport
1,mm_1,576,4,"/mm/deburr, /mm/drill, /mm/mill, /mm/transport_from_to"
2,hw_1,522,1,/hw/human_review
3,ov_2,330,1,/ov/burn
4,wt_2,330,1,/wt/pick_up_and_transport
5,vgr_2,885,1,/vgr/pick_up_and_transport
6,hbw_2,1581,2,"/hbw/store_empty_bucket, /hbw/unload"
7,sm_2,309,2,"/sm/sort, /sm/transport"
8,sm_1,378,2,"/sm/sort, /sm/transport"
9,dm_2,177,3,"/dm/cylindrical_drill, /dm/drill, /dm/lower"


Load a single subevent log file

In [16]:


# Process 0a0a7c16-85d9-48be-a7d5-32931240c337.xes file
filename = "0a0a7c16-85d9-48be-a7d5-32931240c337.xes"
file_path = os.path.join(xes_directory, filename)
print(f"Processing {filename}")

try:
    subevent_log = xes_importer.apply(file_path)
    print("\n\n")
    print(f"Imported {filename} with {len(subevent_log)} traces.")

    # Print important information about the subevent_log
    print(f"Number of events: {sum(len(trace) for trace in subevent_log)}")
    activities = set(event["concept:name"] for trace in subevent_log for event in trace if "concept:name" in event)
    print(f"Number of unique activities: {len(activities)}")
    print(f"Unique activities: {activities}")
    case_ids = [trace.attributes["concept:name"] for trace in subevent_log if "concept:name" in trace.attributes]
    print(f"Number of cases: {len(case_ids)}")
    print(f"First 5 case IDs: {case_ids[:5]}")
    print("\n\n")

except Exception as e:
    print(f"Error processing {filename}: {e}")

Processing 0a0a7c16-85d9-48be-a7d5-32931240c337.xes


parsing log, completed traces :: 100%|██████████| 1/1 [00:00<00:00, 144.08it/s]




Imported 0a0a7c16-85d9-48be-a7d5-32931240c337.xes with 1 traces.
Number of events: 5
Number of unique activities: 5
Unique activities: {'milling the workpiece', 'transporting the workpiece to the sorting machine', 'ejecting the workpiece to the conveyor belt', 'transporting the workpiece to the mill', 'transporting the workpiece to the ejection position'}
Number of cases: 0
First 5 case IDs: []





List all the attributes in this file

In [17]:
try:
    # List all attributes in the subevent_log in a table and display nicely in Jupyter Notebook
    all_attributes = set()
    for trace in subevent_log:
        all_attributes.update(trace.attributes.keys())
        for event in trace:
            all_attributes.update(event.keys())

    # Prepare attribute type information
    attr_info = []
    for attr in all_attributes:
        if attr in subevent_log[0].attributes:
            attr_type = type(subevent_log[0].attributes[attr]).__name__
        elif len(subevent_log[0]) > 0 and attr in subevent_log[0][0]:
            attr_type = type(subevent_log[0][0][attr]).__name__
        else:
            attr_type = "unknown"
        attr_info.append({"Attribute": attr, "Type": attr_type})

    # Display as a pandas DataFrame
    attr_df = pd.DataFrame(attr_info)
    display(attr_df)
    # Save the attribute DataFrame to an Excel file in a "tables" subfolder
    tables_dir = os.path.join(os.getcwd(), "tables")
    os.makedirs(tables_dir, exist_ok=True)
    attr_df.to_excel(os.path.join(tables_dir, "sub_attribute_info.xlsx"), index=False)

except Exception as e:
    print(f"Error processing the subevent_log: {e}")

,Attribute,Type
0,time:timestamp,datetime
1,operation_end_time,datetime
2,concept:name,str
3,stream:datastream,dict
4,SubProcessID,str
5,org:resource,str


Load the database file with the sensor data and query it using DuckDB

In [18]:
import duckdb
import os
from pathlib import Path
import pandas as pd


# Make sure xes_directory is defined before this line
parquet_dir = os.path.join(xes_directory, "parquet")

# Connect to an in-memory DuckDB instance
con = duckdb.connect(database=':memory:')

# Collect all .parquet files in the directory
parquet_file = os.path.join(parquet_dir, "all_combined_new.parquet")

# Ensure there are files to process
if not os.path.exists(parquet_file):
    raise FileNotFoundError(f"No Parquet file found at {parquet_file}")

# Register all Parquet files as a single virtual table
parquet_list_str = f"'{parquet_file}'"
query = f"CREATE VIEW sensor_data AS SELECT * FROM parquet_scan([{parquet_list_str}])"
con.execute(query)

# Print the number of rows in the sensor_data table
if False:
    row_count = con.execute("SELECT COUNT(*) FROM sensor_data").fetchone()[0]
    print(f"Number of rows in sensor_data: {row_count}")

# Print column names
if False:
    info_df = con.execute("PRAGMA table_info('sensor_data')").df()
    print("\nColumns and types in sensor_data:")
    print(info_df['name'])

# Show a sample of the data
if False:
    print("\nSample rows from sensor_data:")
    sample_df = con.execute("SELECT * FROM sensor_data LIMIT 5").df()
    display(sample_df)


# ------------------------
# Enter custom query here
# ------------------------

df_sensor_grouped = con.execute("""
    SELECT 
        *
    FROM sensor_data
    WHERE "org:resource" = 'pm_1'
    AND "concept:name" = 'transporting the workpiece to the punch'
    LIMIT 1
""").df()

# ------------------------


# Display with full width

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
display(df_sensor_grouped)




,trace:SubProcessID,concept:name,org:resource,time:timestamp,operation_end_time,stream:datastream,stream:system,stream:system_type,stream:observation,stream:procedure_type,stream:interaction_type,stream:timestamp,stream:value,sensor_key
0,02d2094a-7ffa-4a52-9c91-5fcd55593308,transporting the workpiece to the punch,pm_1,2021-06-23T15:48:58.201000,2021-06-23T15:49:44.201000,<NA>,http://iot.uni-trier.de/FTOnto#PM_1,sosa:Sensor,http://iot.uni-trier.de/StreamDataAnnotationOnto#PM_1_Property_Current_State,stream:discrete,sosa:Observation,2021-06-23T15:48:58.201000,not ready,stream:point


Number of processes per resource

In [19]:
df_sensor_grouped = con.execute("""
    SELECT 
        "org:resource",
        count(*) AS observation_count,
        SUM(data_points_count) AS total_data_points
        
    FROM 
        (SELECT 
            "org:resource",
            "trace:SubProcessID",
            count(*) AS data_points_count
        FROM sensor_data
        GROUP BY "org:resource", "trace:SubProcessID", 
        )
    GROUP BY "org:resource"
""").df()

# ------------------------


# Display with full width

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
display(df_sensor_grouped)

,org:resource,observation_count,total_data_points
0,vgr_1,575,49035459.0
1,dm_2,59,81183.0
2,hbw_2,506,1833662.0
3,sm_1,119,6239204.0
4,vgr_2,293,847392.0
5,mm_1,189,7454870.0
6,hbw_1,275,54117983.0
7,hw_1,170,194413.0
8,wt_2,110,235296.0
9,ov_1,197,513777.0


Analyse some example process from MainProcess file

In [20]:
# Process MainProcess.xes file
filename = "MainProcess.xes"
file_path = os.path.join(xes_directory, filename)
print(f"Processing {filename}")

try:
    log = xes_importer.apply(file_path)
    print("\n\n")
    print(f"Imported {filename} with {len(log)} traces.")
    
    
    # Display only the concept:name attribute of the events in the first trace
    if len(log) > 0:
        first_trace = log[0]
        print(f"First trace case ID: {first_trace.attributes.get('concept:name', 'N/A')}")
        for idx, event in enumerate(first_trace):
            print(f"Event {idx+1}: concept:name = {event.get('concept:name', 'N/A')}, SubProcessID = {event.get('SubProcessID', 'N/A')}")
    else:
        print("Log is empty, no traces to display.")

    print("\n---\n")
    
    first_subprocess_id = "27958fc0-4484-41ff-9260-e76f8a83a7cd"
    file = first_subprocess_id + ".xes"
    file_path = os.path.join(xes_directory, file)
    
    print(f"Processing subprocess file: {file}")

    sub_log = xes_importer.apply(file_path)
    print(f"Imported {file} with {len(sub_log)} traces.")

    # Display only the concept:name attribute of the events in the first trace of the subprocess log
    if len(sub_log) > 0:
        first_sub_trace = sub_log[0]
        for idx, event in enumerate(first_sub_trace):
            print(f"Event {idx+1}: concept:name = {event.get('concept:name', 'N/A')}, org:resource = {event.get('org:resource', 'N/A')}")
    else:
        print("Subprocess log is empty, no traces to display.")

except Exception as e:
    print(f"Error processing {filename}: {e}")

Processing MainProcess.xes


parsing log, completed traces :: 100%|██████████| 301/301 [00:00<00:00, 680.74it/s]





Imported MainProcess.xes with 301 traces.
First trace case ID: WF_101_0
Event 1: concept:name = /hbw/unload, SubProcessID = N/A
Event 2: concept:name = /hbw/unload, SubProcessID = 27958fc0-4484-41ff-9260-e76f8a83a7cd
Event 3: concept:name = /hbw/unload, SubProcessID = N/A
Event 4: concept:name = /vgr/pick_up_and_transport, SubProcessID = N/A
Event 5: concept:name = /vgr/pick_up_and_transport, SubProcessID = 4d198444-6633-4218-b1f7-ca67ec666360
Event 6: concept:name = /vgr/pick_up_and_transport, SubProcessID = N/A
Event 7: concept:name = /hbw/store_empty_bucket, SubProcessID = N/A
Event 8: concept:name = /hbw/store_empty_bucket, SubProcessID = e73aa303-98af-407e-be4e-5ebdee33bc4d
Event 9: concept:name = /hbw/store_empty_bucket, SubProcessID = N/A
Event 10: concept:name = /vgr/pick_up_and_transport, SubProcessID = N/A
Event 11: concept:name = /vgr/pick_up_and_transport, SubProcessID = 0e7b5a4c-4c03-47b2-96fd-e401ed7fbca9
Event 12: concept:name = /vgr/pick_up_and_transport, SubProcessI

parsing log, completed traces :: 100%|██████████| 1/1 [00:00<00:00, 24.10it/s]

Imported 27958fc0-4484-41ff-9260-e76f8a83a7cd.xes with 1 traces.
Event 1: concept:name = moving towards the slot 0, org:resource = hbw_2
Event 2: concept:name = picking up the bucket from the slot, org:resource = hbw_2
Event 3: concept:name = transporting the bucket to the conveyor belt, org:resource = hbw_2
Event 4: concept:name = dropping off the bucket at the conveyor belt, org:resource = hbw_2
Event 5: concept:name = transporting the bucket to the vacuum gripper robot crane jib, org:resource = hbw_2


Analyze Subprocess Data

In [21]:
df_sensor_grouped = con.execute("""
    SELECT 
        *
    FROM sensor_data
    WHERE "org:resource" = 'ov_1'
    AND "concept:name" = 'transporting the workpiece to the inside of the oven'
    AND "trace:SubProcessID" = '001618a4-1be3-406c-bd83-ebe5cff5e4f7'
    
""").df()

grouped = df_sensor_grouped.groupby(['stream:system', 'stream:observation'])

for name, group in grouped:
    proc_type = group['stream:procedure_type'].iloc[0]
    print(f"\nGroup: {name}")
    if proc_type == 'stream:continuous':
        # Convert stream:value to numeric if possible
        vals = pd.to_numeric(group['stream:value'], errors='coerce')
        min_val = vals.min()
        max_val = vals.max()
        print(f"  stream:procedure_type = stream:continuous, range: [{min_val}, {max_val}]")
    else:
        unique_vals = group['stream:value'].unique()
        print(f"  stream:procedure_type = {proc_type}, values: {unique_vals}")
        


Group: ('http://iot.uni-trier.de/FTOnto#OV_1', 'http://iot.uni-trier.de/FTOnto#OV_1_WT_1_Temperature')
  stream:procedure_type = stream:continuous, range: [25.427, 25.427]

Group: ('http://iot.uni-trier.de/FTOnto#OV_1', 'http://iot.uni-trier.de/StreamDataAnnotationOnto#OV_1_Property_Current_State')
  stream:procedure_type = stream:discrete, values: <ArrowStringArray>
['not ready']
Length: 1, dtype: str

Group: ('http://iot.uni-trier.de/FTOnto#OV_1', 'http://iot.uni-trier.de/StreamDataAnnotationOnto#OV_1_Property_Current_Task_Elapsed_Seconds_Since_Start')
  stream:procedure_type = stream:continuous, range: [3.031237, 5.53083]

Group: ('http://iot.uni-trier.de/FTOnto#OV_1_Light_Barrier_5', 'http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted')
  stream:procedure_type = stream:binary, values: <ArrowStringArray>
['1.0', '0.0']
Length: 2, dtype: str

Group: ('http://iot.uni-trier.de/FTOnto#OV_1_Motor_1', 'http://iot.uni-trier.de/FTOnto#MotorSpeed')
  stream:procedure_type = stream:conti

In [22]:
df_sensor_grouped = con.execute("""
    SELECT 
        *
    FROM sensor_data
    WHERE "org:resource" = 'pm_1'
    AND "concept:name" = 'transporting the workpiece to the punch'
    
""").df()

grouped = df_sensor_grouped.groupby(['stream:system', 'stream:observation'])

for name, group in grouped:
    proc_type = group['stream:procedure_type'].iloc[0]
    print(f"\nGroup: {name}")
    if proc_type == 'stream:continuous':
        # Convert stream:value to numeric if possible
        vals = pd.to_numeric(group['stream:value'], errors='coerce')
        min_val = vals.min()
        max_val = vals.max()
        print(f"  stream:procedure_type = stream:continuous, range: [{min_val}, {max_val}]")
    else:
        unique_vals = group['stream:value'].unique()
        print(f"  stream:procedure_type = {proc_type}, values: {unique_vals}")


Group: ('http://iot.uni-trier.de/FTOnto#PM_1', 'http://iot.uni-trier.de/StreamDataAnnotationOnto#PM_1_Property_Current_State')
  stream:procedure_type = stream:discrete, values: <ArrowStringArray>
['not ready']
Length: 1, dtype: str

Group: ('http://iot.uni-trier.de/FTOnto#PM_1', 'http://iot.uni-trier.de/StreamDataAnnotationOnto#PM_1_Property_Current_Task_Elapsed_Seconds_Since_Start')
  stream:procedure_type = stream:continuous, range: [0.093731, 124.781221]

Group: ('http://iot.uni-trier.de/FTOnto#PM_1_Capacitive_Sensor_6', 'http://iot.uni-trier.de/FTOnto#Electric_Field_Changed')
  stream:procedure_type = stream:binary, values: <ArrowStringArray>
['0.0', '1.0']
Length: 2, dtype: str

Group: ('http://iot.uni-trier.de/FTOnto#PM_1_Capacitive_Sensor_7', 'http://iot.uni-trier.de/FTOnto#Electric_Field_Changed')
  stream:procedure_type = stream:binary, values: <ArrowStringArray>
['0.0', '1.0']
Length: 2, dtype: str

Group: ('http://iot.uni-trier.de/FTOnto#PM_1_Capacitive_Sensor_8', 'http://

Output all the directly follows relations for each resource (used for model creation)

In [23]:
all_resources = ["hbw_1", "hbw_2", "mm_1", "sm_1", "wt_2", "ov_1", "vgr_1", "ov_2", "mm_2", "dm_2", "pm_1", "wt_1", "sm_2", "vgr_2", "hw_1"]

for resource in all_resources:
    df_sensor_grouped = con.execute(f"""
        SELECT 
            "trace:SubProcessID", "time:timestamp", "concept:name"
        FROM sensor_data
        WHERE "org:resource" = '{resource}'
          AND "concept:name" NOT ILIKE 'calibrating%'
        GROUP BY "trace:SubProcessID", "time:timestamp", "concept:name"
    """).df()

    # Group by SubProcessID and collect event names in order
    subprocess_groups = df_sensor_grouped.groupby('trace:SubProcessID')

    events = []

    for sub_id, group in subprocess_groups:
        # Sort by timestamp and get event names
        event_names = group.sort_values('time:timestamp')['concept:name'].tolist()
        # Remove consecutive duplicates
        for i in range(1, len(event_names)):
            if event_names[i] != event_names[i-1] and (event_names[i-1], event_names[i]) not in events:
                events.append((event_names[i-1], event_names[i]))

    # Print all the directly-follows relations
    print(f"\n--- Resource: {resource} ---")
    print(f"Total unique directly-follows relations: {len(events)}")
    num_concept_names = df_sensor_grouped['concept:name'].nunique()
    print(f"Number of unique concept:name values: {num_concept_names}")
    print("Directly-follows relations (event1 -> event2):")
    for e1, e2 in events:
        print(f"  {e1} -> {e2}")



--- Resource: hbw_1 ---
Total unique directly-follows relations: 37
Number of unique concept:name values: 26
Directly-follows relations (event1 -> event2):
  transporting the bucket to the high-bay warehouse crane jib -> transporting the bucket to the high bay warehouse crane jib
  transporting the bucket to the high bay warehouse crane jib -> picking up the bucket from the conveyor belt
  picking up the bucket from the conveyor belt -> dropping off the bucket at the slot
  moving towards the slot 0 -> picking up the bucket from the slot
  picking up the bucket from the slot -> transporting the bucket to the conveyor belt
  transporting the bucket to the conveyor belt -> dropping off the bucket at the conveyor belt
  dropping off the bucket at the conveyor belt -> transporting the bucket to the vacuum gripper robot crane jib
  picking up the bucket from the conveyor belt -> transporting the bucket to the slot 0
  transporting the bucket to the slot 0 -> dropping off the bucket at the 

Find als sensors with discrete values

In [24]:
df_synthetic_data = con.execute("""
    SELECT 
        "stream:observation",
        COUNT(*) AS value_count
    FROM sensor_data
    WHERE "stream:procedure_type" = 'stream:discrete'
    GROUP BY "stream:observation", "stream:procedure_type"
    ORDER BY value_count
""").df()

display(df_synthetic_data)

,stream:observation,value_count
0,http://iot.uni-trier.de/StreamDataAnnotationOnto#DM_2_Property_Current_State,5768
1,http://iot.uni-trier.de/StreamDataAnnotationOnto#SM_2_Property_Current_State,8192
2,http://iot.uni-trier.de/StreamDataAnnotationOnto#MM_2_Property_Current_State,8211
3,http://iot.uni-trier.de/StreamDataAnnotationOnto#PM_1_Property_Current_State,9291
4,http://iot.uni-trier.de/StreamDataAnnotationOnto#SM_1_Property_Current_State,10180
5,http://iot.uni-trier.de/StreamDataAnnotationOnto#MM_1_Property_Current_State,10704
6,http://iot.uni-trier.de/StreamDataAnnotationOnto#OV_2_Property_Current_State,18246
7,NaN,18372
8,http://iot.uni-trier.de/StreamDataAnnotationOnto#WT_2_Property_Current_State,29412
9,http://iot.uni-trier.de/StreamDataAnnotationOnto#OV_1_Property_Current_State,34509


Check the amount of nan values per resource

In [25]:
df_synthetic_data = con.execute("""
    SELECT 
        "org:resource",
        "stream:value",
        "stream:observation",
        "stream:system",
        COUNT(*) AS value_count
    FROM sensor_data
    WHERE "stream:value" IS NULL
       OR lower(trim("stream:value")) IN ('none', 'nan', 'null', '')
    GROUP BY "org:resource", "stream:value", "stream:observation", "stream:system"
    ORDER BY value_count
""").df()

# Here it is shown that the number of missing values is equal to the number of all values (all values are missing for these observations)
df = con.execute("""
    SELECT 
        "org:resource",
        "stream:observation",
        "stream:system",
        COUNT(*) AS value_count
    FROM sensor_data
    WHERE "stream:observation" = 'http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted'
        AND "org:resource" IN ('pm_1', 'dm_2')
    GROUP BY "org:resource", "stream:observation", "stream:system"
    ORDER BY value_count
""").df()

display(df_synthetic_data)
display(df)

,org:resource,stream:value,stream:observation,stream:system,value_count
0,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_1,222
1,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_2,719
2,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_3,919
3,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Waiting_Platform,1018
4,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_4,1027
5,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_5,1134
6,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_6,1201
7,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_7,1243
8,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_8,1280
9,hbw_1,None,http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_9,1291


,org:resource,stream:observation,stream:system,value_count
0,dm_2,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,http://iot.uni-trier.de/FTOnto#DM_2_Light_Barrier_5,5768
1,dm_2,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,http://iot.uni-trier.de/FTOnto#DM_2_Light_Barrier_6,5768
2,dm_2,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,http://iot.uni-trier.de/FTOnto#DM_2_Light_Barrier_7,5768
3,dm_2,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,http://iot.uni-trier.de/FTOnto#DM_2_Light_Barrier_4,5768
4,pm_1,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,http://iot.uni-trier.de/FTOnto#PM_1_Light_Barrier_6,9291
5,pm_1,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,http://iot.uni-trier.de/FTOnto#PM_1_Light_Barrier_4,9291
6,pm_1,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,http://iot.uni-trier.de/FTOnto#PM_1_Light_Barrier_7,9291
7,pm_1,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,http://iot.uni-trier.de/FTOnto#PM_1_Light_Barrier_5,9291


Find synchronized sensors for a resource

In [ ]:
results = []

for resource in all_resources:
    df = con.execute(f"""
        SELECT 
            "stream:timestamp",
            STRING_AGG(
                DISTINCT "stream:system" || ':' || "stream:observation",
                ', ' ORDER BY "stream:system" || ':' || "stream:observation"
            ) AS sensor_group,
            COUNT(DISTINCT "stream:system" || ':' || "stream:observation") AS num_sensors,
            COUNT(*) AS occurrence_count
        FROM (
            SELECT *
            FROM sensor_data
            WHERE "org:resource" = '{resource}'
            LIMIT 20000
        )
        GROUP BY "stream:timestamp"
        ORDER BY occurrence_count DESC
    """).df()

    if not df.empty:
        df_grouped = df.groupby('sensor_group').agg(
            timestamp_count=('stream:timestamp', 'count'),
            total_occurrences=('occurrence_count', 'sum'),
            num_sensors=('num_sensors', 'first')
        ).reset_index().sort_values('timestamp_count', ascending=False)
        
        print(f"\n--- Resource: {resource} ---")
        display(df_grouped)
        
        # Second table: time intervals analysis
        df_intervals = con.execute(f"""
            SELECT 
            "stream:system" || ':' || "stream:observation" AS sensor,
            COUNT(*) AS total_readings,
            AVG(EXTRACT(EPOCH FROM (timestamp_lead - timestamp_value))) AS avg_interval_seconds,
            STDDEV_POP(EXTRACT(EPOCH FROM (timestamp_lead - timestamp_value))) AS stddev_interval_seconds,
            MIN(EXTRACT(EPOCH FROM (timestamp_lead - timestamp_value))) AS min_interval_seconds,
            MAX(EXTRACT(EPOCH FROM (timestamp_lead - timestamp_value))) AS max_interval_seconds
            FROM (
            SELECT 
                "stream:system",
                "stream:observation",
                "time:timestamp",
                CAST("stream:timestamp" AS TIMESTAMP) AS timestamp_value,
                LEAD(CAST("stream:timestamp" AS TIMESTAMP)) OVER (
                PARTITION BY "stream:system", "stream:observation", "time:timestamp"
                ORDER BY CAST("stream:timestamp" AS TIMESTAMP)
                ) AS timestamp_lead
            FROM sensor_data
            WHERE "org:resource" = '{resource}'
              AND "stream:timestamp" IS NOT NULL
            )
            WHERE timestamp_lead IS NOT NULL
            GROUP BY "stream:system", "stream:observation"
            ORDER BY avg_interval_seconds DESC
        """).df()
        
        print(f"\nTime Intervals Analysis for {resource}:")
        display(df_intervals)



--- Resource: hbw_1 ---


,sensor_group,timestamp_count,total_occurrences,num_sensors
0,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2:http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration,10220,10220,1
3,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_Gyroscope_2:http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Rotation,4800,4800,1
4,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_Magnetometer_2:http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Magnetic_Field_Strength,4466,4466,1
5,"http://iot.uni-trier.de/FTOnto#HBW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HBW_1_Crane_Jib_Property_Current_Position_X, http://iot.uni-trier.de/FTOnto#HBW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HBW_1_Crane_Jib_Property_Current_Position_Y, http://iot.uni-trier.de/FTOnto#HBW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HBW_1_Crane_Jib_Property_Target_Position_X, http://iot.uni-trier.de/FTOnto#HBW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HBW_1_Crane_Jib_Property_Target_Position_Y, http://iot.uni-trier.de/FTOnto#HBW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HBW_1_Property_Current_State, http://iot.uni-trier.de/FTOnto#HBW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HBW_1_Property_Current_Task_Elapsed_Seconds_Since_Start, http://iot.uni-trier.de/FTOnto#HBW_1_Light_Barrier_1:http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted, http://iot.uni-trier.de/FTOnto#HBW_1_Light_Barrier_2:http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted, http://iot.uni-trier.de/FTOnto#HBW_1_Light_Barrier_3:http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted, http://iot.uni-trier.de/FTOnto#HBW_1_Light_Barrier_4:http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted, http://iot.uni-trier.de/FTOnto#HBW_1_Motor_1:http://iot.uni-trier.de/FTOnto#MotorSpeed, http://iot.uni-trier.de/FTOnto#HBW_1_Motor_2:http://iot.uni-trier.de/FTOnto#MotorSpeed, http://iot.uni-trier.de/FTOnto#HBW_1_Motor_3:http://iot.uni-trier.de/FTOnto#MotorSpeed, http://iot.uni-trier.de/FTOnto#HBW_1_Motor_4:http://iot.uni-trier.de/FTOnto#MotorSpeed, http://iot.uni-trier.de/FTOnto#HBW_1_Position_Switch_5:http://iot.uni-trier.de/FTOnto#PositionSwitchPressed, http://iot.uni-trier.de/FTOnto#HBW_1_Position_Switch_6:http://iot.uni-trier.de/FTOnto#PositionSwitchPressed, http://iot.uni-trier.de/FTOnto#HBW_1_Position_Switch_7:http://iot.uni-trier.de/FTOnto#PositionSwitchPressed, http://iot.uni-trier.de/FTOnto#HBW_1_Position_Switch_8:http://iot.uni-trier.de/FTOnto#PositionSwitchPressed",27,486,18
2,"http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2:http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration, http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_Magnetometer_2:http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Magnetic_Field_Strength",7,14,2
1,"http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2:http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration, http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_Gyroscope_2:http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Rotation",2,4,2
6,"http://iot.uni-trier.de/FTOnto#HBW_1_Slot_1:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Slot_2:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Slot_3:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Slot_4:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Slot_5:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Slot_6:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Slot_7:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Slot_8:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Slot_9:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID, http://iot.uni-trier.de/FTOnto#HBW_1_Waiting_Platform:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID",1,10,10



Time Intervals Analysis for hbw_1:


,sensor,total_readings,avg_interval_seconds,stddev_interval_seconds,min_interval_seconds,max_interval_seconds
0,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_5:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
1,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_7:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
2,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_1:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
3,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_6:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
4,http://iot.uni-trier.de/FTOnto#HBW_1_Waiting_Platform:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
5,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_3:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
6,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_4:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
7,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_8:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
8,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_2:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07
9,http://iot.uni-trier.de/FTOnto#HBW_1_Slot_9:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,2187,9839.688877,430292.921481,0.046,2.011786e+07



--- Resource: hbw_2 ---


...

,sensor,total_readings,avg_interval_seconds,stddev_interval_seconds,min_interval_seconds,max_interval_seconds
0,http://iot.uni-trier.de/FTOnto#VGR_2:http://iot.uni-trier.de/StreamDataAnnotationOnto#VGR_2_Crane_Jib_Property_Target_Position_Y,52961,406.309972,87462.906667,0.0,2.011807e+07
1,http://iot.uni-trier.de/FTOnto#VGR_2_Motor_1:http://iot.uni-trier.de/FTOnto#MotorSpeed,52961,406.309972,87462.906667,0.0,2.011807e+07
2,http://iot.uni-trier.de/FTOnto#VGR_2:http://iot.uni-trier.de/StreamDataAnnotationOnto#VGR_2_Crane_Jib_Property_Target_Position_X,52961,406.309972,87462.906667,0.0,2.011807e+07
3,http://iot.uni-trier.de/FTOnto#VGR_2_Compressor_7:http://iot.uni-trier.de/FTOnto#CompressorPowerLevel,52961,406.309972,87462.906667,0.0,2.011807e+07
4,http://iot.uni-trier.de/FTOnto#VGR_2_Motor_2:http://iot.uni-trier.de/FTOnto#MotorSpeed,52961,406.309972,87462.906667,0.0,2.011807e+07
5,http://iot.uni-trier.de/FTOnto#VGR_2_Valve_8:http://iot.uni-trier.de/FTOnto#ValveOpen,52961,406.309972,87462.906667,0.0,2.011807e+07
6,http://iot.uni-trier.de/FTOnto#VGR_2:http://iot.uni-trier.de/StreamDataAnnotationOnto#VGR_2_Crane_Jib_Property_Target_Position_Z,52961,406.309972,87462.906667,0.0,2.011807e+07
7,http://iot.uni-trier.de/FTOnto#VGR_2:http://iot.uni-trier.de/StreamDataAnnotationOnto#VGR_2_Property_Current_Task_Elapsed_Seconds_Since_Start,52961,406.309972,87462.906667,0.0,2.011807e+07
8,http://iot.uni-trier.de/FTOnto#VGR_2_Motor_3:http://iot.uni-trier.de/FTOnto#MotorSpeed,52961,406.309972,87462.906667,0.0,2.011807e+07
9,http://iot.uni-trier.de/FTOnto#VGR_2:http://iot.uni-trier.de/StreamDataAnnotationOnto#VGR_2_Crane_Jib_Property_Current_Position_Y,52961,406.309972,87462.906667,0.0,2.011807e+07



--- Resource: hw_1 ---


,sensor_group,timestamp_count,total_occurrences,num_sensors
0,"http://iot.uni-trier.de/FTOnto#HW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HW_1_Property_Current_State, http://iot.uni-trier.de/FTOnto#HW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HW_1_Property_Current_Task_Elapsed_Seconds_Since_Start, http://iot.uni-trier.de/FTOnto#HW_1_Light_Barrier_8:http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted, http://iot.uni-trier.de/FTOnto#HW_1_Manual_Switch_6:http://iot.uni-trier.de/FTOnto#ManualSwitchPressed, http://iot.uni-trier.de/FTOnto#HW_1_Manual_Switch_7:http://iot.uni-trier.de/FTOnto#ManualSwitchPressed",3981,19905,5
3,http://iot.uni-trier.de/FTOnto#HW_1_Workplace:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,68,68,1
1,"http://iot.uni-trier.de/FTOnto#HW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HW_1_Property_Current_State, http://iot.uni-trier.de/FTOnto#HW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HW_1_Property_Current_Task_Elapsed_Seconds_Since_Start, http://iot.uni-trier.de/FTOnto#HW_1_Light_Barrier_8:http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted, http://iot.uni-trier.de/FTOnto#HW_1_Manual_Switch_6:http://iot.uni-trier.de/FTOnto#ManualSwitchPressed, http://iot.uni-trier.de/FTOnto#HW_1_Manual_Switch_7:http://iot.uni-trier.de/FTOnto#ManualSwitchPressed, http://iot.uni-trier.de/FTOnto#HW_1_Workplace:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID",4,24,6
2,"http://iot.uni-trier.de/FTOnto#HW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HW_1_Property_Current_State, http://iot.uni-trier.de/FTOnto#HW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HW_1_Property_Current_Task_Elapsed_Seconds_Since_Start, http://iot.uni-trier.de/FTOnto#HW_1_Manual_Switch_6:http://iot.uni-trier.de/FTOnto#ManualSwitchPressed",1,3,3



Time Intervals Analysis for hw_1:


,sensor,total_readings,avg_interval_seconds,stddev_interval_seconds,min_interval_seconds,max_interval_seconds
0,http://iot.uni-trier.de/FTOnto#HW_1_Workplace:http://iot.uni-trier.de/FTOnto#NFC_Tag_UID,762,1712.934475,23098.038041,0.047,425013.700
1,http://iot.uni-trier.de/FTOnto#HW_1_Manual_Switch_6:http://iot.uni-trier.de/FTOnto#ManualSwitchPressed,38729,33.717707,3246.188158,0.019,424882.653
2,http://iot.uni-trier.de/FTOnto#HW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HW_1_Property_Current_State,38729,33.717707,3246.188158,0.019,424882.653
3,http://iot.uni-trier.de/FTOnto#HW_1_Manual_Switch_7:http://iot.uni-trier.de/FTOnto#ManualSwitchPressed,38729,33.717707,3246.188158,0.019,424882.653
4,http://iot.uni-trier.de/FTOnto#HW_1_Light_Barrier_8:http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,38729,33.717707,3246.188158,0.019,424882.653
5,http://iot.uni-trier.de/FTOnto#HW_1:http://iot.uni-trier.de/StreamDataAnnotationOnto#HW_1_Property_Current_Task_Elapsed_Seconds_Since_Start,38729,33.717707,3246.188158,0.019,424882.653


Get the form of all multidimensional sensor values (like position)

In [ ]:
df = con.execute("""
    SELECT 
        "stream:value",
        "stream:observation",
        "stream:system"
    FROM sensor_data
    WHERE "stream:procedure_type" = 'stream:continuous'
    GROUP BY "stream:value", "stream:observation", "stream:system", "stream:procedure_type"
""").df()

# Filter to keep only non-numerical values
df['_num'] = pd.to_numeric(df['stream:value'], errors='coerce')
df = df[df['_num'].isna()].drop(columns=['_num'])

display(df)

,stream:value,stream:observation,stream:system
1,"[-0.3662, -31.8995, -15.6713]",http://iot.uni-trier.de/FTOnto#VGR_1_Crane_Jib_Rotation,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_Gyroscope_1
2,"[0.846, -10.556, -1.1248]",http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2
3,"[-0.8268, -10.4214, 0.5672]",http://iot.uni-trier.de/FTOnto#VGR_1_Crane_Jib_Acceleration,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_1
4,"[-2.615, -10.1041, -1.3555]",http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2
5,"[1.9997, -16.853, 4.0378]",http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2
...,...,...,...
30766763,"[0.6441, -10.9886, -0.4807]",http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2
30766764,"[-10.445, 0.9919, -3.7461]",http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Rotation,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_Gyroscope_2
30766765,"[-0.5191, -11.5366, 1.2498]",http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2
30766766,"[0.5384, -8.4313, 0.2115]",http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration,http://iot.uni-trier.de/FTOnto#BMX055_Pi_1_AccSensor_2


List all possible observations of sensors

In [ ]:
df_observations = con.execute("""
    SELECT
        "stream:observation",
        first("stream:value" ORDER BY "stream:timestamp") AS "example value",
        list(DISTINCT "stream:system_type" ORDER BY "stream:system_type") AS system_types,
        list(DISTINCT "stream:procedure_type" ORDER BY "stream:procedure_type") AS procedure_types,
        list(DISTINCT "org:resource" ORDER BY "org:resource") AS resources
    FROM sensor_data
    WHERE "stream:observation" IS NOT NULL
    GROUP BY "stream:observation"
    ORDER BY "stream:observation"
""").df()

pd.set_option('display.max_rows', None)
display(df_observations)



,stream:observation,example value,system_types,procedure_types,resources
0,http://iot.uni-trier.de/FTOnto#CompressorPowerLevel,0.0,[sosa:Actuator],[stream:continuous],"[mm_1, mm_2, ov_1, ov_2, sm_1, sm_2, vgr_1, vgr_2, wt_1, wt_2]"
1,http://iot.uni-trier.de/FTOnto#Electric_Field_Changed,0.0,[sosa:Sensor],[stream:binary],"[dm_2, pm_1]"
2,http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Acceleration,"[0.7691, -9.9695, 0.1923]",[sosa:Sensor],[stream:continuous],[hbw_1]
3,http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Magnetic_Field_Strength,"[49.6682, 17.1785, -9.3233]",[sosa:Sensor],[stream:continuous],[hbw_1]
4,http://iot.uni-trier.de/FTOnto#HBW_1_Crane_Jib_Rotation,"[4.7533, -2.2202, -5.295]",[sosa:Sensor],[stream:continuous],[hbw_1]
5,http://iot.uni-trier.de/FTOnto#LightBarrierInterrupted,0.0,[sosa:Sensor],[stream:binary],"[dm_2, hbw_1, hbw_2, hw_1, mm_1, mm_2, ov_1, ov_2, pm_1, sm_1, sm_2]"
6,http://iot.uni-trier.de/FTOnto#MM_1_Compressor_8_Vibration,"[0.5494, -0.0785, 10.0454]",[sosa:Sensor],[stream:continuous],[mm_1]
7,http://iot.uni-trier.de/FTOnto#MM_1_Motor_3_Vibration,"[0.0, 0.0, 8.1619]",[sosa:Sensor],[stream:continuous],[mm_1]
8,http://iot.uni-trier.de/FTOnto#MM_1_Pneumatic_System_Pressure,671.0696044989999,[sosa:Sensor],[stream:continuous],[mm_1]
9,http://iot.uni-trier.de/FTOnto#MM_1_Temperature,27.577,[sosa:Sensor],[stream:continuous],[mm_1]


Getting the number of values, where Position Sensor 2 is True compared to where Positon sensor 1 is True

In [ ]:
df_sensor_grouped = con.execute("""
    SELECT
        "stream:system",
        "stream:value",
        COUNT(*) AS value_count
    FROM sensor_data
    WHERE "org:resource" = 'ov_1'
        AND "concept:name" = 'transporting the workpiece to the outside of the oven'
        AND "stream:system" ILIKE '%position%'
        AND "stream:value" IS NOT NULL
    GROUP BY "stream:system", "stream:value"
    ORDER BY "stream:system", value_count DESC
""").df()

display(df_sensor_grouped)

,stream:system,stream:value,value_count
0,http://iot.uni-trier.de/FTOnto#OV_1_Position_Switch_1,0.0,2588
1,http://iot.uni-trier.de/FTOnto#OV_1_Position_Switch_1,1.0,227
2,http://iot.uni-trier.de/FTOnto#OV_1_Position_Switch_2,0.0,2743
3,http://iot.uni-trier.de/FTOnto#OV_1_Position_Switch_2,1.0,72


Getting the number of values where position sensor is 0 for tempering the workpiece

In [ ]:
df_sensor_grouped = con.execute("""
    SELECT
        "stream:system",
        "stream:value",
        COUNT(*) AS value_count
    FROM sensor_data
    WHERE "org:resource" = 'ov_1'
        AND "concept:name" = 'temper the workpiece for 40 seconds'
        AND "stream:system" ILIKE '%position%'
        AND "stream:value" IS NOT NULL
    GROUP BY "stream:system", "stream:value"
    ORDER BY "stream:system", value_count DESC
""").df()

display(df_sensor_grouped)

,stream:system,stream:value,value_count
0,http://iot.uni-trier.de/FTOnto#OV_1_Position_Switch_1,1.0,13865
1,http://iot.uni-trier.de/FTOnto#OV_1_Position_Switch_1,0.0,190
2,http://iot.uni-trier.de/FTOnto#OV_1_Position_Switch_2,0.0,14055


In [ ]:
df_subprocess_ids = con.execute("""
    SELECT
        "trace:SubProcessID" AS "stream:SubProcessID",
        COUNT(*) AS matches
    FROM sensor_data
    WHERE "org:resource" = 'ov_1'
      AND "concept:name" = 'temper the workpiece for 40 seconds'
      AND "stream:system" = 'http://iot.uni-trier.de/FTOnto#OV_1_Position_Switch_1'
      AND TRY_CAST("stream:value" AS DOUBLE) = 0.0
    GROUP BY "trace:SubProcessID"
    ORDER BY matches DESC, "trace:SubProcessID"
""").df()

display(df_subprocess_ids)

# Optional: only the distinct SubProcessID values as a Python list
subprocess_ids = df_subprocess_ids["stream:SubProcessID"].dropna().tolist()
print(subprocess_ids)

,stream:SubProcessID,matches
0,082c4fca-2b66-438f-a933-fe1cd4c1b36e,197
1,1e6be670-c66f-42bb-aca2-7c5995fcadb9,197
2,2aee3b24-076d-40ca-ad09-9deb2885d7a1,197
3,3ca03f0e-3c1e-4dbb-bb78-1912b3aa2117,197
4,41531963-a8cb-4f2c-9695-fa3749a6414a,197
5,548415e8-a739-411c-a1aa-aa1dca05d269,197
6,69f24f4d-1851-42d1-b2a1-57ef2aea3fc4,197
7,6fac0fd4-7629-4c87-bcb5-2373812ce14c,197
8,bdaeea16-0f58-4074-ac41-737f1b8210ff,197
9,c1031713-46de-4b94-951b-b56dc1fdd827,197


['082c4fca-2b66-438f-a933-fe1cd4c1b36e', '1e6be670-c66f-42bb-aca2-7c5995fcadb9', '2aee3b24-076d-40ca-ad09-9deb2885d7a1', '3ca03f0e-3c1e-4dbb-bb78-1912b3aa2117', '41531963-a8cb-4f2c-9695-fa3749a6414a', '548415e8-a739-411c-a1aa-aa1dca05d269', '69f24f4d-1851-42d1-b2a1-57ef2aea3fc4', '6fac0fd4-7629-4c87-bcb5-2373812ce14c', 'bdaeea16-0f58-4074-ac41-737f1b8210ff', 'c1031713-46de-4b94-951b-b56dc1fdd827', 'd4de1719-8ba5-4da0-a91a-d239ff43fc1a', 'e451ab30-b7c5-4198-b0eb-cfaae439c066', '019f360f-f22c-4526-8f36-d0aae6cc890c', '0871a3e6-157f-451b-b254-8cbd3b1f0592', '1736b385-d5bf-4090-9739-3aef55e51d48', '1a73f4cd-a4a7-4312-8139-08a8732f3816', '248b1c51-b98f-4785-b8c3-bb0f982d4e8d', '27380990-7b3c-4f2e-9475-23b8ca920e73', '3c97f9e6-22f5-4428-bb8e-c3c2648effdd', '3dd56bb8-cebc-4faf-9c27-68e84192e6bc', '43507866-b7ba-43dd-bd6b-98bc77cb583f', '44b44904-3512-40e4-8870-825eb99ca833', '4d232895-9f9c-4ff0-86f7-9c3944922799', '4e3a470e-d84b-4e9a-850c-97814b628f32', '50513203-25e9-4e8c-a745-84b3d259e24e',